### Alphametiken – Lösung Schritt für Schritt

*33. Bundeswettbewerb Informatik (2014), 1. Runde, Aufgabe 4*

Alphametiken sind Rechenrätsel; das wohl bekannteste Beispiel ist

    SEND + MORE = MONEY

welches ein Globetrotter an seine Heimatadresse geschickt haben soll.

Die Frage ist, wie jeder Buchstabe so durch eine Ziffer ersetzt werden kann, dass die Rechnung aufgeht. Dabei sollen gleiche Buchstaben durch gleiche Ziffern und verschiedene Buchstaben durch verschiedene Ziffern ersetzt werden. Mit etwas Nachdenken lässt sich die Lösung finden: Setzt man O = 0, M = 1, Y = 2, E = 5, N = 6, D = 7, R = 8 und S = 9, dann ergibt sich 9567 + 1085 = 10652.

Auch auf Deutsch gibt es solche Rätsel:

    SUCHEN – MACHT = SPASS

Kannst du die Lösung finden?

#### Aufgabe 4

1. Im Allgemeinen ist eine Alphametik eine Gleichung, die aus Buchstaben und den Operatoren +, –, \* und / besteht. Die Operatoren dürfen nur auf der linken Seite der Gleichung auftreten. Schreibe ein Programm, das eine Alphametik einliest und eine Lösung dafür findet – oder ausgibt, dass keine Lösung existiert.
2. Nun geht es darum, solche Alphametiken selbst zu konstruieren. Wir wollen uns dabei auf „Zahl-Alphametiken“ beschränken, die nur aus deutschen Zahlwörtern bestehen und auch „gelesen“ korrekte Rechnungen beschreiben. Zum Beispiel hat die Zahl-Alphametik

       ZWEI + VIER = SECHS

   unter anderem die Lösungen 4236 + 9635 = 13871 und 8624 + 3427 = 12051. Ein weiteres Beispiel ist:

       EINS + EINS + EINS + EINS = VIER

   Schreibe ein Programm, das systematisch nach Zahl-Alphametiken sucht und diese mitsamt Lösungen ausgibt.

   Kann dein Programm eine Zahl-Alphametik finden, die mehr als 20 Buchstaben enthält?

**Zu den Daten:** Beispieldaten gibt es zu dieser Aufgabe nicht. In `eigene_testdaten/alphametiken.txt` stehen acht Gleichungen, eine pro Zeile: die vier aus der Aufgabe und die „langen Alphametiken“ aus den Lösungshinweisen (dort mit Klammern geschrieben, hier ausmultipliert: aus `3·(−2·ZWEI + FUENF) = DREI` wird `3*FUENF-6*ZWEI=DREI`).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Alles ausprobieren (Brute Force):** Man probiert jede Zuordnung von Ziffern zu Buchstaben und rechnet nach. Weil verschiedene Buchstaben verschiedene Ziffern haben, sind das nicht `10^k` Möglichkeiten, sondern nur `10·9·8·…` (Permutationen).
- **Frühzeitig abbrechen (Backtracking):** Wer die Buchstaben nicht alle auf einmal, sondern einen nach dem anderen vergibt, kann schon nach wenigen Buchstaben erkennen, dass eine Belegung nie zu einer Lösung wird, und spart sich alles, was danach käme. Hier hilft die **letzte Stelle**: Bei `…D + …E = …Y` muss `D + E` auf dieselbe Ziffer wie `Y` enden, lange bevor die vorderen Buchstaben bekannt sind.
- **Rechnen mit dem Rest (`%`):** Dass zwei Zahlen auf die gleichen `j` Stellen enden, heißt: Ihr Unterschied ist durch `10^j` teilbar. Das gilt auch beim Rechnen mit `+`, `-` und `*`.
- **Punkt vor Strich** beim Auswerten einer Gleichung ohne `eval`.

**Nützliche Funktionen und Bibliotheken**
- `itertools.permutations(range(10), k)`: alle Anordnungen von `k` verschiedenen Ziffern. Für Teil 2: `itertools.combinations` und `itertools.product`.
- `set(text)`: die verschiedenen Zeichen (Buchstaben) eines Textes.
- `zip`, `dict`, `sorted`, `str.isdigit()` und `wort[-j:]` (die letzten `j` Zeichen).

## Die Idee in Worten

**Teil 1.** Wir zerlegen die Gleichung in Wörter und Operatoren, setzen für jeden Buchstaben eine Ziffer ein und rechnen nach. Dabei gelten die Regeln der Aufgabe: verschiedene Buchstaben haben verschiedene Ziffern, und ein Wort **beginnt nicht mit 0**. Zuerst probieren wir **alle** Belegungen aus, dann bauen wir ein Verfahren, das die Buchstaben **von der letzten Stelle her** vergibt und früh abbricht. Beide müssen dieselben Lösungen finden.

**Teil 2.** Eine Zahl-Alphametik ist zugleich eine **richtige Rechnung** (`2 + 4 = 6`) und eine **lösbare Alphametik** (`ZWEI + VIER = SECHS`). Wir wählen also Zahlwörter, schreiben links Vielfache davon mit `+` und `-` und rechts ein Wort, prüfen zuerst die Rechnung mit den Zahlen und dann mit dem Verfahren aus Teil 1, ob die Buchstaben-Rechnung lösbar ist. Mehr als 10 verschiedene Buchstaben dürfen es nicht sein, denn es gibt nur 10 Ziffern.

# Teil 1: Eine Alphametik lösen

## Schritt 1: Die Gleichung zerlegen

Wir lesen die Gleichung als Text, zum Beispiel `SUCHEN-MACHT=SPASS`. Leerzeichen und die Zeichen `–` und `·` aus dem Aufgabentext machen wir zu `-` und `*`. Die Funktion gibt die **Wörter** der linken Seite, die **Operatoren** dazwischen und das **Wort der rechten Seite** zurück. Zahlen wie in `4*EINS` bleiben als Text `'4'` unter den Wörtern stehen (so schreibt die Aufgabe die Kurzform `4 · EINS`).

In [ ]:
from itertools import permutations, combinations, product
from math import perm


def zerlege(gleichung):
    text = gleichung.replace(' ', '').replace('–', '-').replace('·', '*')
    links, rechts = text.split('=')
    woerter = []
    operatoren = []
    aktuell = ''
    for zeichen in links:
        if zeichen in '+-*/':
            woerter.append(aktuell)
            operatoren.append(zeichen)
            aktuell = ''
        else:
            aktuell = aktuell + zeichen
    woerter.append(aktuell)
    return woerter, operatoren, rechts


print(zerlege('SUCHEN – MACHT = SPASS'))
print(zerlege('4*EINS+ZWEI=VIER'))

(['SUCHEN', 'MACHT'], ['-'], 'SPASS')
(['4', 'EINS', 'ZWEI'], ['*', '+'], 'VIER')


## Schritt 2: Wörter und Gleichungen auswerten

`wert(wort, belegung)` macht aus einem Wort eine Zahl: Von links nach rechts `zahl = zahl * 10 + ziffer`. Die Belegung ist ein Wörterbuch `Buchstabe → Ziffer`. Mit dem Parameter `stellen` nehmen wir nur die **letzten** Stellen des Wortes (brauchen wir erst in Schritt 4).

`rechne(operatoren, zahlen)` berechnet die linke Seite mit **Punkt vor Strich**, ohne `eval`: `*` und `/` rechnen wir sofort in `aktuell` aus, bei `+` und `-` wird `aktuell` mit seinem Vorzeichen in die Liste der Summanden gelegt. Die Division soll nur gelten, wenn sie **aufgeht**; sonst gibt die Funktion `None` zurück (die Belegung ist dann keine Lösung). Mehrere `*` und `/` rechnen wir von links nach rechts.

In [ ]:
def wert(wort, belegung, stellen=100):
    if wort.isdigit():                          # eine Zahl wie '4' in 4*EINS
        return int(wort)
    zahl = 0
    for buchstabe in wort[-stellen:]:
        zahl = zahl * 10 + belegung[buchstabe]
    return zahl


def rechne(operatoren, zahlen):
    summanden = []
    vorzeichen = 1
    aktuell = zahlen[0]
    for i in range(len(operatoren)):
        op = operatoren[i]
        z = zahlen[i + 1]
        if op == '*':
            aktuell = aktuell * z
        elif op == '/':
            if z == 0 or aktuell % z != 0:
                return None
            aktuell = aktuell // z
        else:                                   # + oder -: der bisherige Summand ist fertig
            summanden.append(vorzeichen * aktuell)
            vorzeichen = 1 if op == '+' else -1
            aktuell = z
    summanden.append(vorzeichen * aktuell)
    return sum(summanden)


bel = {'S': 9, 'E': 5, 'N': 6, 'D': 7, 'M': 1, 'O': 0, 'R': 8, 'Y': 2}
print(wert('SEND', bel), wert('MORE', bel), wert('MONEY', bel))
print(rechne(['+'], [9567, 1085]))
print(rechne(['+', '*', '-'], [2, 3, 4, 5]), '(= 2 + 3*4 - 5)')
print(rechne(['/'], [7, 2]), rechne(['/', '+'], [12, 4, 1]))

9567 1085 10652
10652
9 (= 2 + 3*4 - 5)
None 4


## Schritt 3: Alle Belegungen ausprobieren

Aus der Gleichung lesen wir die verschiedenen **Buchstaben** ab (ohne die Zahlen) und die **Anfangsbuchstaben** der Wörter (sie dürfen nicht 0 sein). Dann gehen wir mit `permutations(range(10), k)` alle Möglichkeiten durch, `k` verschiedene Ziffern für `k` Buchstaben, und rechnen jedes Mal nach. Mit `fuehrende_null=True` darf auch eine 0 vorn stehen. Die Funktion gibt alle Lösungen und die Zahl der tatsächlich geprüften Belegungen zurück.

Wie viele Belegungen sind das? Bei `SEND + MORE = MONEY` gibt es 8 Buchstaben, davon 2 Anfangsbuchstaben (`S` und `M`). Die Anfangsbuchstaben wählen wir aus den Ziffern 1 bis 9 (9·8 Möglichkeiten), die übrigen 6 Buchstaben aus den restlichen 8 Ziffern (`8!/2!` Möglichkeiten): zusammen 9·8·20160 = 1 451 520, wie in den Hinweisen. `anzahl_versuche` rechnet das mit `math.perm(n, k)` (= `n!/(n-k)!`) aus. `zeige` setzt die Ziffern in die Gleichung ein.

In [ ]:
def loese_brute_force(gleichung, fuehrende_null=False):
    woerter, operatoren, rechts = zerlege(gleichung)
    alle = [w for w in woerter + [rechts] if not w.isdigit()]
    buchstaben = sorted(set(''.join(alle)))
    anfaenge = set()
    if not fuehrende_null:
        anfaenge = {w[0] for w in alle}
    loesungen = []
    versuche = 0
    for ziffern in permutations(range(10), len(buchstaben)):
        belegung = dict(zip(buchstaben, ziffern))
        if any(belegung[a] == 0 for a in anfaenge):
            continue
        versuche = versuche + 1
        zahlen = [wert(w, belegung) for w in woerter]
        if rechne(operatoren, zahlen) == wert(rechts, belegung):
            loesungen.append(belegung)
    return loesungen, versuche


def anzahl_versuche(gleichung):
    woerter, operatoren, rechts = zerlege(gleichung)
    alle = [w for w in woerter + [rechts] if not w.isdigit()]
    k = len(set(''.join(alle)))
    a = len({w[0] for w in alle})
    anzahl = perm(10 - a, k - a)                # die Buchstaben, die nicht vorn stehen
    for i in range(a):
        anzahl = anzahl * (9 - i)               # die Anfangsbuchstaben: 9, 8, ... Möglichkeiten
    return anzahl


def zeige(gleichung, belegung):
    text = gleichung.replace(' ', '').replace('–', '-').replace('·', '*')
    return ''.join(str(belegung.get(zeichen, zeichen)) for zeichen in text)


print(anzahl_versuche('SEND+MORE=MONEY'), anzahl_versuche('SUCHEN-MACHT=SPASS'))
loesungen, versuche = loese_brute_force('EINS+EINS=ZWEI')
print(versuche, 'Belegungen geprüft,', len(loesungen), 'Lösungen, zum Beispiel', zeige('EINS+EINS=ZWEI', loesungen[0]))
loesungen, versuche = loese_brute_force('ABC+DEF=ABC')
print(len(loesungen), 'Lösungen' if loesungen else '-> keine Lösung')

1451520 2903040
120960 Belegungen geprüft, 11 Lösungen, zum Beispiel 1407+1407=2814
0 -> keine Lösung


`EINS + EINS = ZWEI` hat nur 6 Buchstaben und nur rund 120 000 Belegungen, das geht schnell. Bei `SEND + MORE = MONEY` (1,45 Millionen) und besonders bei `SUCHEN – MACHT = SPASS` (2,9 Millionen Belegungen) dauert das Durchprobieren schon einige Sekunden, die Gleichung mit der Auswertung wird ja jedes Mal neu berechnet. Und bei der Division, die wir nicht umgehen können, bleibt uns das Ausprobieren ohnehin. Je ein Beispiel mit `/` und mit `*` (die Gleichungen haben mehrere Lösungen, wir zeigen die erste und die letzte):

In [ ]:
for gleichung in ('ABC/D=EF', 'A*B=CD'):
    loesungen, versuche = loese_brute_force(gleichung)
    print(gleichung, ':', len(loesungen), 'Lösungen, zum Beispiel', zeige(gleichung, loesungen[0]), 'und', zeige(gleichung, loesungen[-1]))

ABC/D=EF : 76 Lösungen, zum Beispiel 106/2=53 und 760/8=95
A*B=CD : 40 Lösungen, zum Beispiel 2*5=10 und 9*8=72


## Schritt 4: Schneller – von der letzten Stelle her

Die Idee: Schreibt man die Wörter untereinander, so bestimmt die **letzte Spalte** (die Einerstellen) schon allein, ob die Gleichung aufgehen *kann*. Bei `SEND + MORE = MONEY` muss `D + E` auf dieselbe Ziffer wie `Y` enden. Wenn das nicht stimmt, ist es egal, welche Ziffern `S`, `M` und `O` bekommen.

Genauer: Rechnet man nur mit `+`, `-` und `*`, so hängen die letzten `j` Stellen des Ergebnisses nur von den **letzten `j` Stellen der Wörter** ab. Die Gleichung `links = rechts` verlangt insbesondere, dass `links - rechts` durch `10^j` teilbar ist, für jedes `j`. Das prüfen wir mit `%`, sobald alle Buchstaben aus den letzten `j` Stellen aller Wörter eine Ziffer haben.

Das Verfahren:
1. Wir legen eine **Reihenfolge der Buchstaben** fest: zuerst die Buchstaben der letzten Stelle jedes Wortes, dann die der vorletzten usw. Dabei merken wir uns, nach welchem Buchstaben welches `j` geprüft werden kann (`pruefen_nach`).
2. Wir vergeben die Ziffern **rekursiv** (`suche(k)`: der `k`-te Buchstabe der Reihenfolge), nur freie Ziffern und keine 0 für Anfangsbuchstaben. Nach jeder Vergabe prüfen wir den Rest für die fälligen `j`. Passt er nicht, probieren wir die nächste Ziffer (wir **brechen früh ab**).
3. Sind alle Buchstaben vergeben, prüfen wir die Gleichung **exakt**. Die Prüfung mit dem Rest allein reicht dafür nicht: Sie sagt nur, dass `links - rechts` durch `10^j` teilbar ist, nicht, dass es 0 ist.

Mit der Division geht das nicht (zwei Zahlen mit gleichen Endstellen können sehr verschieden dividiert werden), dafür bleibt das Durchprobieren; `loese` wählt das passende Verfahren. Der Parameter `nur_erste` bricht bei der ersten Lösung ab (brauchen wir in Teil 2). Als Maß für den Aufwand zählen wir die vergebenen Ziffern (`knoten`).

In [ ]:
def loese_schnell(gleichung, fuehrende_null=False, nur_erste=False):
    woerter, operatoren, rechts = zerlege(gleichung)
    alle = [w for w in woerter + [rechts] if not w.isdigit()]
    anfaenge = set()
    if not fuehrende_null:
        anfaenge = {w[0] for w in alle}
    # Reihenfolge der Buchstaben: erst alle letzten Stellen, dann alle vorletzten, ...
    reihenfolge = []
    pruefen_nach = {}                       # Index in der Reihenfolge -> Liste der j, die jetzt geprüft werden können
    for j in range(1, max(len(w) for w in alle) + 1):
        for w in alle:
            if len(w) >= j and w[-j] not in reihenfolge:
                reihenfolge.append(w[-j])
        k = len(reihenfolge) - 1
        if k not in pruefen_nach:
            pruefen_nach[k] = []
        pruefen_nach[k].append(j)
    loesungen = []
    belegung = {}
    benutzt = [False] * 10
    knoten = [0]                            # Zähler: wie viele Ziffern wurden vergeben

    def suche(k):
        if k == len(reihenfolge):           # alle Buchstaben haben eine Ziffer: exakt prüfen
            zahlen = [wert(w, belegung) for w in woerter]
            if rechne(operatoren, zahlen) == wert(rechts, belegung):
                loesungen.append(dict(belegung))
            return
        buchstabe = reihenfolge[k]
        for ziffer in range(10):
            if benutzt[ziffer] or (ziffer == 0 and buchstabe in anfaenge):
                continue
            benutzt[ziffer] = True
            belegung[buchstabe] = ziffer
            knoten[0] = knoten[0] + 1
            passt = True
            if k in pruefen_nach:
                for j in pruefen_nach[k]:   # die letzten j Stellen müssen stimmen
                    zahlen = [wert(w, belegung, j) for w in woerter]
                    if (rechne(operatoren, zahlen) - wert(rechts, belegung, j)) % 10 ** j != 0:
                        passt = False
                        break
            if passt:
                suche(k + 1)
            benutzt[ziffer] = False
            del belegung[buchstabe]
            if nur_erste and len(loesungen) > 0:
                return

    suche(0)
    return loesungen, knoten[0]


def loese(gleichung, fuehrende_null=False):
    # Mit Division rechnet nur das Durchprobieren richtig, sonst das schnelle Verfahren
    if '/' in gleichung:
        return loese_brute_force(gleichung, fuehrende_null)
    return loese_schnell(gleichung, fuehrende_null)


for gleichung in ('SEND+MORE=MONEY', 'SUCHEN-MACHT=SPASS'):
    loesungen, knoten = loese(gleichung)
    print(gleichung, ':', len(loesungen), 'Lösung(en),', knoten, 'Ziffern vergeben statt', anzahl_versuche(gleichung), 'Belegungen')
    for belegung in loesungen:
        print('  ', zeige(gleichung, belegung), '  ', belegung)

SEND+MORE=MONEY : 1 Lösung(en), 7956 Ziffern vergeben statt 1451520 Belegungen
   9567+1085=10652    {'D': 7, 'E': 5, 'Y': 2, 'N': 6, 'R': 8, 'O': 0, 'S': 9, 'M': 1}
SUCHEN-MACHT=SPASS : 1 Lösung(en), 14168 Ziffern vergeben statt 2903040 Belegungen
   108347-95836=12511    {'N': 7, 'T': 6, 'S': 1, 'E': 4, 'H': 3, 'C': 8, 'A': 5, 'P': 2, 'U': 0, 'M': 9}


Beide Pflichtbeispiele sind gelöst, und zwar eindeutig: `SEND + MORE = MONEY` hat nur die Lösung `9567 + 1085 = 10652`, und `SUCHEN – MACHT = SPASS` hat nur **108347 – 95836 = 12511**, also `S=1, U=0, C=8, H=3, E=4, N=7, M=9, A=5, T=6, P=2` (wie in den Hinweisen). Statt 1,45 bzw. 2,9 Millionen Belegungen wurden nur 7 956 bzw. 14 168 Ziffern vergeben.

## Schritt 5: Probe

Wir prüfen, ob das schnelle Verfahren **genau dieselben** Belegungen findet wie das Durchprobieren. Dafür nehmen wir kleinere Rätsel, weil das Durchprobieren sonst zu lange dauert. Bei sehr kleinen Rätseln wie `AB+BA=CC` werden sogar mehr Ziffern vergeben als es Belegungen gibt (das Verfahren zählt jede Zwischenstufe mit); der Gewinn zeigt sich bei Rätseln mit vielen Buchstaben. (Der Generator dieses Notebooks hat auch `SEND + MORE = MONEY`, `SUCHEN – MACHT = SPASS` und `ZWEI + VIER = SECHS` auf beide Arten gerechnet, mit demselben Ergebnis.)

In [ ]:
kleine = ['EINS+EINS=ZWEI', 'EINS+ZWEI=DREI', 'AB+BA=CC', 'A*B=CD', 'ABC-DEF=GH', 'AB*C-D=EF']
for gleichung in kleine:
    schnell, knoten = loese_schnell(gleichung)
    brute, versuche = loese_brute_force(gleichung)
    gleich = sorted(sorted(b.items()) for b in schnell) == sorted(sorted(b.items()) for b in brute)
    print(gleichung, ':', len(schnell), 'Lösungen,', 'gleiche Lösungen' if gleich else 'UNTERSCHIED', '(', knoten, 'statt', versuche, ')')

EINS+EINS=ZWEI : 11 Lösungen, gleiche Lösungen ( 1036 statt 120960 )
EINS+ZWEI=DREI : 0 Lösungen, gleiche Lösungen ( 676 statt 1270080 )
AB+BA=CC : 32 Lösungen, gleiche Lösungen ( 585 statt 504 )
A*B=CD : 40 Lösungen, gleiche Lösungen ( 929 statt 3528 )
ABC-DEF=GH : 716 Lösungen, gleiche Lösungen ( 32888 statt 1270080 )
AB*C-D=EF : 232 Lösungen, gleiche Lösungen ( 16011 statt 90720 )


Die Hinweise nennen eine weitere Zahl zu `SEND + MORE = MONEY`: Wenn Zahlen mit 0 beginnen dürfen, gibt es **25** Lösungen (ohne die Regel „verschiedene Buchstaben, verschiedene Ziffern“ sogar 605). Die erste Zahl können wir nachprüfen. Und für `ZWEI + VIER = SECHS` suchen wir die beiden Lösungen aus der Aufgabe:

In [ ]:
loesungen, knoten = loese_schnell('SEND+MORE=MONEY', fuehrende_null=True)
print(len(loesungen), 'Lösungen mit führender Null möglich, zum Beispiel', zeige('SEND+MORE=MONEY', loesungen[0]))
loesungen, knoten = loese_schnell('ZWEI+VIER=SECHS')
print(len(loesungen), 'Lösungen für ZWEI + VIER = SECHS, darunter')
for belegung in loesungen:
    if zeige('ZWEI+VIER=SECHS', belegung) in ('4236+9635=13871', '8624+3427=12051'):
        print('  ', zeige('ZWEI+VIER=SECHS', belegung))

25 Lösungen mit führender Null möglich, zum Beispiel 7531+0825=08356
12 Lösungen für ZWEI + VIER = SECHS, darunter
   8624+3427=12051
   4236+9635=13871


Teil 1 ist damit gelöst: Das Programm liest eine Gleichung und gibt alle Lösungen aus, oder (wie bei `ABC+DEF=ABC` oben, wo `DEF` nicht 0 sein kann) die Meldung, dass es keine gibt.

# Teil 2: Zahl-Alphametiken erzeugen

## Schritt 6: Die Zahlwörter

Für Teil 2 brauchen wir die deutschen Zahlwörter. Wir schreiben sie selbst auf (`Zahl → Wort`), in Großbuchstaben und ohne Umlaute: **FUENF**, **ZWOELF**, **DREISSIG** (`ü` wird `ue`, `ö` wird `oe`, `ß` wird `ss`). Wir nehmen die Zahlen von 1 bis 20 und die vollen Zehner bis 100. Eine Bibliothek ist nicht nötig, so sieht man die Schreibweise.

Ein Wort mit mehr als 10 verschiedenen Buchstaben kann keine Alphametik ergeben, und das gilt auch für **Mengen** von Zahlwörtern: Zusammen dürfen sie höchstens 10 verschiedene Buchstaben haben, denn es gibt nur 10 Ziffern. Das prüfen wir mit `set`, bevor wir rechnen.

In [ ]:
ZAHLWOERTER = {1: 'EINS', 2: 'ZWEI', 3: 'DREI', 4: 'VIER', 5: 'FUENF', 6: 'SECHS', 7: 'SIEBEN', 8: 'ACHT', 9: 'NEUN',
               10: 'ZEHN', 11: 'ELF', 12: 'ZWOELF', 13: 'DREIZEHN', 14: 'VIERZEHN', 15: 'FUENFZEHN', 16: 'SECHZEHN',
               17: 'SIEBZEHN', 18: 'ACHTZEHN', 19: 'NEUNZEHN', 20: 'ZWANZIG', 30: 'DREISSIG', 40: 'VIERZIG',
               50: 'FUENFZIG', 60: 'SECHZIG', 70: 'SIEBZIG', 80: 'ACHTZIG', 90: 'NEUNZIG', 100: 'HUNDERT'}

for zahlen in ([1, 2, 3, 4], [1, 2, 3, 4, 5, 6, 7], [1, 2, 3, 4, 5, 6, 7, 8, 9]):
    buchstaben = set()
    for zahl in zahlen:
        buchstaben = buchstaben | set(ZAHLWOERTER[zahl])
    print(len(buchstaben), 'verschiedene Buchstaben in den Zahlwörtern für', zahlen)

9 verschiedene Buchstaben in den Zahlwörtern für [1, 2, 3, 4]
14 verschiedene Buchstaben in den Zahlwörtern für [1, 2, 3, 4, 5, 6, 7]
16 verschiedene Buchstaben in den Zahlwörtern für [1, 2, 3, 4, 5, 6, 7, 8, 9]


Die Zahlwörter von EINS bis NEUN haben zusammen mehr als 10 verschiedene Buchstaben, man kann also nicht alle auf einmal nehmen. Das ist der Grund, warum die Gleichungen mit wenigen Wörtern arbeiten.

## Schritt 7: Systematisch suchen

Wir suchen Gleichungen der Form `c1·W1 + c2·W2 + … = W` mit der Kurzschreibweise aus der Aufgabe (`4 · EINS = VIER`), wobei die Faktoren `ci` ganze Zahlen von `-3` bis `3` (ohne 0) sind. Dazu:

1. `combinations(zahlen, m)`: alle Auswahlen von `m` **verschiedenen** Zahlen (hier `m` = 2 oder 3). Haben die Wörter zusammen mehr als 10 verschiedene Buchstaben, lassen wir die Auswahl aus. In jeder Auswahl ist jede der Zahlen einmal das **rechte** Wort, die übrigen stehen links.
2. `product(faktoren, repeat=…)`: alle Möglichkeiten für die Faktoren der linken Wörter (jeder Faktor kommt auch negativ vor, sonst gäbe es kein `ZWEI − EINS = EINS`).
3. **Zuerst die Rechnung mit den Zahlen:** Stimmt `c1·n1 + c2·n2 + … = n` nicht, ist die Gleichung als Zahlen-Rechnung falsch, und wir lassen sie weg. Das trifft die meisten. Sind alle Faktoren negativ, lassen wir sie auch weg (dieselbe Gleichung mit vertauschten Vorzeichen findet man anders).
4. Dann schreiben wir die Gleichung als Text (positive Faktoren zuerst, `3*FUENF-6*ZWEI=DREI`) und fragen `loese_schnell(…, nur_erste=True)`: Gibt es mindestens eine Belegung der Buchstaben?

Als **Zahl der Buchstaben** zählen wir (wie die Hinweise), wie viele Buchstaben die Gleichung hat, wenn man jedes Vielfache als Summe ausschreibt: `4*EINS=VIER` ist `EINS + EINS + EINS + EINS = VIER`, also 4·4 + 4 = 20 Buchstaben.

In [ ]:
def als_text(terme, rechts):
    # terme: Liste von (Faktor, Zahl); die Gleichung als Text, z. B. 3*FUENF-6*ZWEI=DREI
    text = ''
    for faktor, zahl in terme:
        if faktor < 0:
            text = text + '-'
        elif text != '':
            text = text + '+'
        if abs(faktor) == 1:
            text = text + ZAHLWOERTER[zahl]
        else:
            text = text + str(abs(faktor)) + '*' + ZAHLWOERTER[zahl]
    return text + '=' + ZAHLWOERTER[rechts]


def zaehle_buchstaben(gleichung):
    # So viele Buchstaben hat die Gleichung, wenn man jedes Vielfache als Summe ausschreibt
    woerter, operatoren, rechts = zerlege(gleichung)
    anzahl = len(rechts)
    faktor = 1
    for i in range(len(woerter)):
        if woerter[i].isdigit():
            faktor = faktor * int(woerter[i])
        else:
            anzahl = anzahl + faktor * len(woerter[i])
        if i < len(operatoren) and operatoren[i] in '+-':
            faktor = 1
    return anzahl


def suche_zahl_alphametiken(bis, max_faktor, max_woerter):
    zahlen = [n for n in ZAHLWOERTER if n <= bis]
    faktoren = [f for f in range(-max_faktor, max_faktor + 1) if f != 0]
    rechnungen = 0                                  # Gleichungen, die als Rechnung stimmen
    gefunden = []                                   # Zahl-Alphametiken als Text
    for m in range(2, max_woerter + 1):
        for auswahl in combinations(zahlen, m):
            if len(set(''.join(ZAHLWOERTER[n] for n in auswahl))) > 10:
                continue                            # mehr als 10 Buchstaben: nie lösbar
            for rechts in auswahl:
                links = [n for n in auswahl if n != rechts]
                for fs in product(faktoren, repeat=len(links)):
                    summe = 0
                    for f, n in zip(fs, links):
                        summe = summe + f * n
                    if summe != rechts or max(fs) < 0:
                        continue
                    rechnungen = rechnungen + 1
                    terme = sorted(zip(fs, links), reverse=True)        # positive Faktoren zuerst
                    text = als_text(terme, rechts)
                    if len(loese_schnell(text, nur_erste=True)[0]) > 0:
                        gefunden.append(text)
    return rechnungen, gefunden


rechnungen, gefunden = suche_zahl_alphametiken(12, 3, 3)
print(rechnungen, 'Gleichungen stimmen als Rechnung,', len(gefunden), 'davon sind lösbare Alphametiken, zum Beispiel:')
for text in gefunden[:8]:
    print(' ', text)

370 Gleichungen stimmen als Rechnung, 122 davon sind lösbare Alphametiken, zum Beispiel:
  2*EINS=ZWEI
  3*EINS=DREI
  2*ZWEI=VIER
  3*ZWEI=SECHS
  3*DREI=NEUN
  2*VIER=ACHT
  2*SECHS=ZWOELF
  3*ZWEI-3*EINS=DREI


Mit den Zahlen von 1 bis 12, Faktoren von -3 bis 3 und höchstens zwei linken Wörtern stimmen 370 Gleichungen als Rechnung, und davon sind 122 auch als Buchstabenrätsel lösbar. Die ersten sind die Vielfachen (`2*EINS=ZWEI`, `3*EINS=DREI`, …), danach kommen Gleichungen mit zwei Wörtern links. Zu jeder gehört mindestens eine Belegung. Hier die ersten Gleichungen mit zwei verschiedenen Wörtern links, jeweils mit einer Lösung (die Faktoren stehen dabei weiter als `3*`, `6*` vor der Zahl):

In [ ]:
zeige_anzahl = 0
for text in gefunden:
    if '+' in text or '-' in text:                  # mindestens zwei verschiedene Wörter links
        loesungen, knoten = loese_schnell(text, nur_erste=True)
        print(text, '  zum Beispiel:', zeige(text, loesungen[0]))
        zeige_anzahl = zeige_anzahl + 1
        if zeige_anzahl == 8:
            break

3*ZWEI-3*EINS=DREI   zum Beispiel: 3*6051-3*5134=2751
VIER-2*EINS=ZWEI   zum Beispiel: 9830-2*3846=2138
3*ZWEI-2*EINS=VIER   zum Beispiel: 3*3710-2*1056=9018
2*EINS+ZWEI=VIER   zum Beispiel: 2*2301+4723=9325
FUENF-2*ZWEI=EINS   zum Beispiel: 17401-2*6249=4903
3*ZWEI-FUENF=EINS   zum Beispiel: 3*9410-27162=1068
FUENF-3*EINS=ZWEI   zum Beispiel: 15281-3*2984=6329
3*ZWEI-EINS=FUENF   zum Beispiel: 3*9410-1068=27162


Die Aufgabe nennt auch `ZWEI + VIER = SECHS` (steht in der Liste, als `VIER+ZWEI=SECHS`: Zahlen 2, 4 und 6, alle Faktoren 1) und `EINS + EINS + EINS + EINS = VIER`. Die zweite ist `4*EINS=VIER`, hat den Faktor 4 und kommt in unserer Suche mit Faktoren bis 3 nicht vor.

## Schritt 8: Mehr als 20 Buchstaben?

**Ja.** Die Zahl der Buchstaben hängt von den Faktoren ab: `3*ZWEI-3*EINS=DREI` ist `3·ZWEI − 3·EINS = DREI` und hat 3·4 + 3·4 + 4 = 28 Buchstaben, wenn man sie als Summe von Wörtern ausschreibt. Allein in unserer kleinen Suche haben 50 der 122 gefundenen Zahl-Alphametiken mehr als 20 Buchstaben, die längste (`3*FUENF-3*EINS=ZWOELF`) hat 33. Mit größeren Faktoren wird es beliebig lang; die Beispiele aus den Hinweisen gehen bis 571 Buchstaben. Wir lesen sie aus der Datei und lösen sie mit demselben Programm (Schritte 1 bis 4):

In [ ]:
gleichungen = []
for zeile in open('eigene_testdaten/alphametiken.txt'):
    if zeile.strip():
        gleichungen.append(zeile.strip())

for gleichung in gleichungen:
    loesungen, knoten = loese(gleichung)
    print(gleichung, ':', zaehle_buchstaben(gleichung), 'Buchstaben,', len(loesungen), 'Lösung(en), zum Beispiel', zeige(gleichung, loesungen[0]))

SEND+MORE=MONEY : 13 Buchstaben, 1 Lösung(en), zum Beispiel 9567+1085=10652
SUCHEN-MACHT=SPASS : 16 Buchstaben, 1 Lösung(en), zum Beispiel 108347-95836=12511
ZWEI+VIER=SECHS : 13 Buchstaben, 12 Lösung(en), zum Beispiel 5432+8239=13671
EINS+EINS+EINS+EINS=VIER : 20 Buchstaben, 1 Lösung(en), zum Beispiel 1329+1329+1329+1329=5316
3*FUENF-6*ZWEI=DREI : 43 Buchstaben, 6 Lösung(en), zum Beispiel 3*16201-6*7529=3429
4*FUENF-8*ZWEI=VIER : 56 Buchstaben, 6 Lösung(en), zum Beispiel 4*16831-8*7280=9084
7*FUENF-14*ZWEI=SIEBEN : 97 Buchstaben, 1 Lösung(en), zum Beispiel 7*52095-14*1704=340809
90*VIER-45*ZEHN+9*ELF=NEUN : 571 Buchstaben, 4 Lösung(en), zum Beispiel 90*3241-45*6407+9*458=7497


Das sind die vier „langen Alphametiken“ der Hinweise mit **43, 56, 97 und 571 Buchstaben**. `SIEBEN` ist dabei eindeutig lösbar, die drei anderen haben mehrere Lösungen (6, 6 und 4). Dass die Lösung des Rätsels mit 571 Buchstaben so schnell gefunden wird, liegt daran, dass alle diese Rätsel nur **9 oder 10 verschiedene Buchstaben** haben. Die Zahl der Buchstaben, die man beim Ausschreiben zählt, spielt für den Aufwand kaum eine Rolle, entscheidend ist die Zahl der verschiedenen. Im Programm steckt das in `wert`: `90*VIER` wird als `90 * 4-stellige Zahl` gerechnet, nicht als 90 einzelne Summanden.

## Schritt 9: Größere Suche und ihre Grenzen

Die Suche oben haben wir bei den Zahlen bis 12 gestoppt, damit sie schnell bleibt. Mit allen Zahlen bis 20 (und denselben Faktoren und höchstens zwei Wörtern links) stimmen 952 Gleichungen als Rechnung, davon sind 142 lösbar. Wer `suche_zahl_alphametiken(20, 3, 3)` aufruft, wartet entsprechend länger. Das Laufzeitproblem, das die Hinweise ansprechen, kommt aus den Möglichkeiten:
- Mit jeder zusätzlichen Zahl in der Auswahl wächst die Zahl der Auswahlen (`combinations`) und mit jedem zusätzlichen Wort links die Zahl der Faktorfolgen (`product`) um ein Vielfaches.
- Jede Gleichung, die als Rechnung stimmt, muss mit dem Verfahren aus Teil 1 untersucht werden (bis zu einigen zehntausend vergebenen Ziffern).

Zwei Dinge halten die Suche in Grenzen: Auswahlen mit mehr als 10 verschiedenen Buchstaben werden sofort aussortiert, und die billige Rechnung mit den Zahlen kommt vor dem Verteilen der Buchstaben.

## Zum Schluss: Geht das auch mit MiniZinc?

**Ja, sehr gut.** Alphametiken sind ein Musterbeispiel für Probleme, die man mit **Bedingungen** statt mit einem Verfahren beschreibt: Jeder Buchstabe ist eine Variable mit einer Ziffer von 0 bis 9, `alldifferent` verlangt verschiedene Ziffern, Anfangsbuchstaben sind größer als 0, und die Gleichung ist eine Summe. Das gilt für alle Gleichungen mit `+`, `-` und Zahlen als Faktoren (`4*EINS`), also für alle acht Beispiele. Für `*` zwischen zwei Wörtern oder `/` wäre das Modell nichtlinear und umständlicher; wir lassen es weg.

Damit das Modell klein bleibt, rechnen wir die Gleichung vorher zu **einer** Summe um: `links - rechts = 0`. Jeder Buchstabe bekommt einen Beiwert `c[i]`: Jedes Vorkommen im Wort an der Stelle `10^p` zählt `10^p` (mal Vorzeichen und Faktor), auf der rechten Seite mit negativem Vorzeichen. Bei `SEND + MORE = MONEY` hat `D` den Beiwert 1 (aus `SEND`), `E` den Beiwert 100 (aus `SEND`) + 1 (aus `MORE`) − 10 (aus `MONEY`) = 91, `M` den Beiwert −9000 (aus `MONEY` und `MORE`: 1000 − 10000) usw. Eine Python-Zelle `als_dzn` macht daraus die Datendatei.

```minizinc
include "alldifferent.mzn";

int: n;                              % Anzahl der verschiedenen Buchstaben
array[1..n] of int: c;               % Beiwert jedes Buchstabens in der Gleichung "links - rechts = 0"
set of int: anfang;                  % Buchstaben, die vorn stehen (dürfen nicht 0 sein)

array[1..n] of var 0..9: x;          % die Ziffer jedes Buchstabens

constraint alldifferent(x);          % verschiedene Buchstaben, verschiedene Ziffern
constraint forall(i in anfang)(x[i] > 0);
constraint sum(i in 1..n)(c[i] * x[i]) = 0;

solve satisfy;
output ["x = \(x)\n"];
```

Die Daten für `SEND + MORE = MONEY` (`alphametik.dzn`, Buchstaben in alphabetischer Reihenfolge `D E M N O R S Y`):

```minizinc
n = 8;
c = [1, 91, -9000, -90, -900, 10, 1000, -1];
anfang = {3, 7};
```

Wir lassen den Solver **alle** Lösungen suchen (in der MiniZinc IDE: „User-defined behavior“ → „Print all solutions“, auf der Kommandozeile `-a`). **Ausgabe** (`x` in der Reihenfolge der Buchstaben, hier `D=7, E=5, M=1, N=6, O=0, R=8, S=9, Y=2`):

```
x = [7, 5, 1, 6, 0, 8, 9, 2]
----------
==========
```

Zum Vergleich mit Python rechnen wir die Daten aus den Beispielen um:

In [ ]:
def als_dzn(gleichung):
    woerter, operatoren, rechts = zerlege(gleichung)
    buchstaben = sorted(set(''.join(w for w in woerter + [rechts] if not w.isdigit())))
    beiwert = {b: 0 for b in buchstaben}
    # Terme der linken Seite: (Vorzeichen, Faktoren des Terms), ein Term ist z. B. 4*EINS
    terme = []
    vorzeichen = 1
    faktoren = [woerter[0]]
    for op, wort in zip(operatoren, woerter[1:]):
        if op == '*':
            faktoren.append(wort)
        else:
            terme.append((vorzeichen, faktoren))
            vorzeichen = 1 if op == '+' else -1
            faktoren = [wort]
    terme.append((vorzeichen, faktoren))
    for vorzeichen, faktoren in terme:
        faktor = vorzeichen
        for f in faktoren:
            if f.isdigit():
                faktor = faktor * int(f)
        wort = [f for f in faktoren if not f.isdigit()][0]
        for p, buchstabe in enumerate(reversed(wort)):
            beiwert[buchstabe] = beiwert[buchstabe] + faktor * 10 ** p
    for p, buchstabe in enumerate(reversed(rechts)):
        beiwert[buchstabe] = beiwert[buchstabe] - 10 ** p
    anfang = sorted(set(buchstaben.index(w[0]) + 1 for w in woerter + [rechts] if not w.isdigit()))
    werte = [beiwert[b] for b in buchstaben]
    return buchstaben, 'n = %d;\nc = %s;\nanfang = {%s};' % (len(buchstaben), werte, ', '.join(str(i) for i in anfang))


buchstaben, dzn = als_dzn('SEND+MORE=MONEY')
print(buchstaben)
print(dzn)
buchstaben, dzn = als_dzn('3*FUENF-6*ZWEI=DREI')
print(buchstaben)
print(dzn)

['D', 'E', 'M', 'N', 'O', 'R', 'S', 'Y']
n = 8;
c = [1, 91, -9000, -90, -900, 10, 1000, -1];
anfang = {3, 7};
['D', 'E', 'F', 'I', 'N', 'R', 'U', 'W', 'Z']
n = 9;
c = [-1000, 230, 30003, -7, 30, -100, 3000, -600, -6000];
anfang = {1, 3, 9};


**Messung bei allen acht Beispielen** (alle Lösungen, jeweils unter einer Sekunde, mit allen drei Solvern Gecode, Chuffed und CP-SAT; die Zahl der Lösungen ist in allen Fällen gleich der von Python, der Generator hat auch die Belegungen selbst verglichen):

| Gleichung | Buchstaben | Lösungen (Python) | Lösungen (MiniZinc) |
|---|---|---|---|
| `SEND+MORE=MONEY` | 8 | 1 | 1 |
| `SUCHEN-MACHT=SPASS` | 10 | 1 | 1 |
| `ZWEI+VIER=SECHS` | 9 | 12 | 12 |
| `EINS+EINS+EINS+EINS=VIER` | 6 | 1 | 1 |
| `3*FUENF-6*ZWEI=DREI` | 9 | 6 | 6 |
| `4*FUENF-8*ZWEI=VIER` | 9 | 6 | 6 |
| `7*FUENF-14*ZWEI=SIEBEN` | 9 | 1 | 1 |
| `90*VIER-45*ZEHN+9*ELF=NEUN` | 10 | 4 | 4 |

Das Rätsel mit 571 Buchstaben ist für MiniZinc nicht schwerer als `SEND + MORE = MONEY`: Es hat nur zehn Variablen, die großen Beiwerte (zum Beispiel 90 000 für `V`) stören nicht. Das ist der Vorteil: Wir mussten die Reihenfolge der Buchstaben und die Prüfung mit den Endstellen (Schritt 4) nicht selbst bauen, der Solver erledigt das Abwägen mit **Propagation** (er streicht aus den Wertebereichen der Variablen alles, was nach den Bedingungen nicht mehr möglich ist).

Die Grenze ist die Form der Gleichung: Mit `*` zwischen zwei Wörtern oder `/` müsste man Wörter als eigene Variablen mit Werten einführen (`SEND = 1000*S + 100*E + 10*N + D`) und Produkte über Variablen rechnen. Für Teil 2 (die Suche nach Zahl-Alphametiken) braucht man MiniZinc nicht, dort ist die Schleife über Zahlwörter und Faktoren das Problem, nicht die einzelne Gleichung; die einzelne Gleichung kann man danach mit MiniZinc prüfen.

## Für Neugierige

- **Größter gemeinsamer Teiler:** Die Hinweise suchen die Faktoren nicht durch Ausprobieren, sondern mit dem **erweiterten euklidischen Algorithmus**: Damit sich die Zahlwörter links zum Wort rechts ergänzen lassen, muss der ggT der linken Zahlen die rechte Zahl teilen; die Faktoren liefert der Algorithmus. Wir haben stattdessen alle Faktoren von -3 bis 3 durchprobiert, das ist für Anfänger verständlicher und reicht für kleine Zahlen. Wer größere Faktoren will, kommt mit dem euklidischen Algorithmus schneller.
- **Alle Teilmengen:** Die Hinweise erzeugen die Mengen der Zahlen mit den Bits einer Laufvariablen `i` (Bit `k` gesetzt heißt: die Zahl `k+1` ist dabei). Mit `combinations` ist das dieselbe Idee in Kurzform.
- **Mehr Wörter links:** Setze `max_woerter` auf 4 (drei Wörter links) und suche in den Zahlen bis 12. Es dauert länger; schau dir an, wie stark `rechnungen` wächst.
- **Division und Produkte zwischen Wörtern:** `loese` kann mit `/` und `*` zwischen Wörtern umgehen; probiere zum Beispiel `ABCD/EF=GH`. (Dass die Division nur dann zählt, wenn sie aufgeht, ist unsere Festlegung, die Aufgabe sagt dazu nichts.)
- **Eigene Rätsel:** Eine gute Alphametik hat genau **eine** Lösung. Suche in `gefunden` die Zahl-Alphametiken mit nur einer Lösung (`len(loese_schnell(text)[0]) == 1`).